## Matplotlib M+L Flaws Robustness Study

In [ ]:
# === Robustness scoring for GitHub Matplotlib code ===
# Creates 1/0 columns for each rule and writes a new CSV.

import os, ast, re, sys
import pandas as pd
import google.generativeai as genai
import time

LLM_MIN_INTERVAL = float(os.getenv("LLM_MIN_INTERVAL", "6.0"))  # seconds; ~10 calls/min by default
_LLM_LAST = 0.0  # do not touch elsewhere

# ---- Gemini (robust setup) ----
HAVE_GEMINI = False
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")  # change if you really need another

try:
    import google.generativeai as genai  # pip install google-generativeai
    # Prefer env var; fallback to local module if you keep one.
    api_key = os.getenv("GOOGLE_API_KEY", None)
    if not api_key:
        from GEMINI_API_KEY import GEMINI_API_KEY as api_key  # requires a GEMINI_API_KEY.py with that name
    if not api_key:
        raise RuntimeError("No Gemini API key found in env or GEMINI_API_KEY.py")

    genai.configure(api_key=api_key)
    client = genai.GenerativeModel(MODEL_NAME)
    HAVE_GEMINI = True
except Exception as e:
    print(f"⚠️ Gemini disabled: {e}")
    HAVE_GEMINI = False


# ----------------- Non-contextual rule detector (your logic) -----------------
def find_noncontextual_flaws(mpl_file: str):
    flaws = []

    # Check if matplotlib is imported
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return flaws

    scan_text = mpl_file[mpl_index:]

    # Descriptive Labels
    for fn in ["title", "xlabel", "ylabel"]:
        match = re.search(rf"{fn}\s*\(\s*['\"]([^'\"]*)['\"]", scan_text)
        if match:
            txt = match.group(1).strip().lower()
            if not txt or txt in ["x", "y", "series 1"]:
                flaws.append(f"MISSING_{fn.upper()}")
        else:
            flaws.append(f"MISSING_{fn.upper()}")

    # Legend required when multiple series plotted
    plot_count = len(re.findall(r'plot\s*\(', scan_text))
    scatter_count = len(re.findall(r'scatter\s*\(', scan_text))
    if (plot_count + scatter_count > 1) and 'legend(' not in scan_text:
        flaws.append("MISSING_LEGEND")

    # Minimum Font Size (>= 15)
    font_matches = re.findall(r'fontsize\s*=\s*(\d+)', scan_text)
    if any(int(size) < 15 for size in font_matches):
        flaws.append("FONTSIZE_TOO_SMALL")

    # Minimum Figure Size (>= 8x5)
    fig_match = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_match:
        w, h = float(fig_match.group(1)), float(fig_match.group(2))
        if w < 8 or h < 5:
            flaws.append("FIGSIZE_TOO_SMALL")

    # High-contrast colors only
    color_matches = re.findall(r'color\s*=\s*[\'"]([^\'"]+)[\'"]', scan_text)
    safe_colors = {
        "#000000", "#0072B2", "#009E73", "#D55E00",
        "black", "blue", "green", "orange"
    }
    if any(color.lower() not in safe_colors for color in color_matches):
        flaws.append("INSUFFICIENT_COLOR_CONTRAST")

    # No animations allowed
    if "FuncAnimation" in scan_text or "animation." in scan_text:
        flaws.append("ANIMATIONS")

    # Inverted Y-axis
    if re.search(r'\.\s*invert_yaxis\s*\(', scan_text):
        flaws.append("INVERTED_Y_AXIS")

    # Truncated Y-axis (not starting at 0)
    for match in re.finditer(r'(?:set_)?ylim\s*\(\s*([\-]?\d+(?:\.\d+)?)\s*,', scan_text):
        lower = float(match.group(1))
        if abs(lower) > 1e-6:
            flaws.append("TRUNCATED_Y_AXIS")
            break

    # 3D Effects
    if (re.search(r'["\']\s*3d\s*["\']', scan_text) or
        "Axes3D" in scan_text or
        "plot_surface(" in scan_text):
        flaws.append("3D_EFFECTS")

    # Tampered aspect ratio
    fig_aspect = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_aspect:
        w, h = float(fig_aspect.group(1)), float(fig_aspect.group(2))
        if h != 0:
            ratio = w / h
            if ratio < 0.5 or ratio > 2.0:
                flaws.append("TAMPERED_ASPECT_RATIO")
    if re.search(r'set_aspect\s*\(|aspect\s*=', scan_text):
        flaws.append("TAMPERED_ASPECT_RATIO")

    # Dual Y-axes
    if re.search(r'twin[xy]\s*\(', scan_text) or 'secondary_y=True' in scan_text:
        flaws.append("DUAL_Y_AXES")
    
    return list(sorted(set(flaws)))

# ----------------- Contextual rule detector (LLM) -----------------
def generate_response(prompt: str) -> str:
    result = client.generate_content(prompt)
    candidate = result.candidates[0]
    return candidate.content.parts[0].text.strip()

_CONTEXTUAL_RULES_PROMPT_TEMPLATE = """You are an expert in data visualization integrity. I will provide you with:

1. A list of misleading visualization rules (each with a RULE_CODE and description),
2. A Matplotlib code snippet that generates a chart.

Your task:
- Analyze the code and detect which rules are violated based solely on what can be inferred from the code itself (e.g., axis behavior, titles, aspect ratio, annotations).
- Output only the list of violated RULE_CODEs in exactly this format: ["RULE_CODE1", "RULE_CODE2", ...]
- If the graph does not violate any rules, return: NONE

Rule Codes and Descriptions:

BIASED_TITLE:
A graph uses a biased or emotionally slanted title that influences interpretation before data is analyzed.

MISLEADING_ANNOTATIONS:
Annotations suggest causality or relationships that are not statistically or contextually justified.

DECEPTIVE_LABELS:
Y-axis or x-axis labels are vague, reversed, or omit key categories, leading to confusion.

FRAMING_BIAS:
External context or textual framing (e.g., comments, hashtags, plot subtitles) introduces bias not reflected in the graph.

INVERTED_AXES:
Y-axis is reversed (top to bottom), which misleads users by flipping the meaning of increases/decreases.

TRUNCATED_AXES:
Y-axis does not start at zero, which exaggerates visual differences.

ASPECT_RATIO_DISTORTION:
Aspect ratio is altered (e.g., too stretched or squished), making trends look steeper or flatter than they are.

DUAL_AXES:
Chart uses two different y-axes that may falsely suggest correlation between unrelated data series.

NON_SEQUENTIAL_AXIS:
X or Y axis uses a non-logical or out-of-order sequence (e.g., age ranges like 18-34, 45-55, 35-44).

Matplotlib Code:
{code}
"""

def find_contextual_flaws(mpl_file: str):
    # Return [] if no matplotlib found
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return []

    scan_text = mpl_file[mpl_index:]

    if not HAVE_GEMINI:
        return []  # gracefully skip if Gemini isn't configured

    prompt = _CONTEXTUAL_RULES_PROMPT_TEMPLATE.format(code=scan_text)
    # --- minimal throttle + soft handling of 429/quota ---
    global _LLM_LAST
    elapsed = time.time() - _LLM_LAST
    if elapsed < LLM_MIN_INTERVAL:
        time.sleep(LLM_MIN_INTERVAL - elapsed)

    try:
        raw = generate_response(prompt)   # your function stays the same
        _LLM_LAST = time.time()
    except Exception as e:
        msg = str(e)
        if "ResourceExhausted" in msg or "429" in msg or "quota" in msg.lower() or "rate" in msg.lower():
            # simple backoff, then skip this row so the run keeps going
            time.sleep(LLM_MIN_INTERVAL * 1.5)
            return []
    raise

    if raw.strip().upper() == "NONE":
        return []

    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            # normalize to strings and unique
            return list(sorted(set(str(x).strip().upper() for x in parsed)))
    except Exception:
        # one retry with a stricter instruction if parse fails
        prompt2 = prompt + "\nReturn ONLY a valid Python list literal of strings like [\"RULE\", \"RULE2\"]."
        raw2 = generate_response(prompt2)
        if raw2.strip().upper() == "NONE":
            return []
        try:
            parsed2 = ast.literal_eval(raw2)
            if isinstance(parsed2, list):
                return list(sorted(set(str(x).strip().upper() for x in parsed2)))
        except Exception:
            pass
    # If still messy, bail out quietly
    return []

# ----------------- Columns to emit -----------------
NONCONTEXTUAL_CODES = [
    "MISSING_TITLE",
    "MISSING_XLABEL",
    "MISSING_YLABEL",
    "MISSING_LEGEND",
    "FONTSIZE_TOO_SMALL",
    "FIGSIZE_TOO_SMALL",
    "INSUFFICIENT_COLOR_CONTRAST",
    "ANIMATIONS",
    "INVERTED_Y_AXIS",
    "TRUNCATED_Y_AXIS",
    "3D_EFFECTS",
    "TAMPERED_ASPECT_RATIO",
    "DUAL_Y_AXES",
]

CONTEXTUAL_CODES = [
    "BIASED_TITLE",
    "MISLEADING_ANNOTATIONS",
    "DECEPTIVE_LABELS",
    "FRAMING_BIAS",
    "INVERTED_AXES",
    "TRUNCATED_AXES",
    "ASPECT_RATIO_DISTORTION",
    "DUAL_AXES",
    "NON_SEQUENTIAL_AXIS",
]

ALL_RULE_COLS = NONCONTEXTUAL_CODES + CONTEXTUAL_CODES

# ----------------- Load CSV (with or without .csv) -----------------
src = 'github_matplotlib_audit_1'
fname = src if os.path.exists(src) else (src + '.csv' if os.path.exists(src + '.csv') else src)

# If you used df = pd.read_csv('github_matplotlib_audit_1'), this keeps behavior identical:
try:
    df = pd.read_csv(fname)
except Exception:
    # last attempt: force add .csv
    df = pd.read_csv(src + '.csv')
    fname = src + '.csv'

print(f"Loaded: {fname}  (rows={len(df)})")

# ----------------- Find the column that holds code -----------------
POSSIBLE_CODE_COLS = [
    "matplotlib_code", "code", "file_content", "content", "snippet", "source", "body", "text"
]
code_col = None
for c in POSSIBLE_CODE_COLS:
    if c in df.columns:
        code_col = c
        break
if code_col is None:
    # as a fallback, pick the first object dtype column with long-ish text
    for c in df.columns:
        if df[c].dtype == object:
            # peek at a few rows
            sample = df[c].dropna().astype(str).head(10)
            if sample.str.contains("import matplotlib|from matplotlib", regex=True, case=False).any():
                code_col = c
                break
if code_col is None:
    raise ValueError("Could not find a column with Matplotlib code. "
                     "Add a column named e.g., 'matplotlib_code' or update POSSIBLE_CODE_COLS.")

print(f"Using code column: {code_col}")

# ----------------- Initialize 1/0 columns -----------------
for col in ALL_RULE_COLS:
    if col not in df.columns:
        df[col] = 0

# Optional: keep string lists of which rules fired
if "NONCONTEXTUAL_LIST" not in df.columns:
    df["NONCONTEXTUAL_LIST"] = ""
if "CONTEXTUAL_LIST" not in df.columns:
    df["CONTEXTUAL_LIST"] = ""

# ----------------- Score each row -----------------
def score_row(code: str):
    if not isinstance(code, str):
        return [], []
    nonctx = find_noncontextual_flaws(code)
    ctx = find_contextual_flaws(code) if HAVE_GEMINI else []
    return nonctx, ctx

for idx, row in df.iterrows():
    code = row[code_col]
    nonctx_list, ctx_list = score_row(code)

    # set 1/0 flags
    for r in NONCONTEXTUAL_CODES:
        if r in nonctx_list:
            df.at[idx, r] = 1
    for r in CONTEXTUAL_CODES:
        if r in ctx_list:
            df.at[idx, r] = 1

    # store lists (semicolon-separated)
    df.at[idx, "NONCONTEXTUAL_LIST"] = ";".join(nonctx_list)
    df.at[idx, "CONTEXTUAL_LIST"] = ";".join(ctx_list)

# ----------------- Save scored CSV -----------------
base, ext = os.path.splitext(fname)
out = f"{base}_scored_2{ext or '.csv'}"
df.to_csv(out, index=False)
print(f"✅ Wrote: {out}")


C:\Users\Administrator\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: github_matplotlib_audit_1.csv  (rows=14956)
Using code column: code


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 56
}
]

In [2]:
import google.generativeai as genai; genai.__version__


'0.8.5'

In [2]:
df1 = pd.read_csv("github_matplotlib_audit_1_scored.csv")


In [3]:
df1.head()

,repo,filename,pushed_date,code,MISSING_TITLE,MISSING_XLABEL,MISSING_YLABEL,MISSING_LEGEND,FONTSIZE_TOO_SMALL,FIGSIZE_TOO_SMALL,...,MISLEADING_ANNOTATIONS,DECEPTIVE_LABELS,FRAMING_BIAS,INVERTED_AXES,TRUNCATED_AXES,ASPECT_RATIO_DISTORTION,DUAL_AXES,NON_SEQUENTIAL_AXIS,NONCONTEXTUAL_LIST,CONTEXTUAL_LIST
0,PoldiB/Matplotlib,Muenchen Wetter.py,2022-01-01,import pandas as pd\nfrom matplotlib import py...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,INSUFFICIENT_COLOR_CONTRAST,NaN
1,VivianYuan12138/python_matplotlib_learning,Python可视化教程.md,2022-01-01,# Python可视化教程\n\n### Matplotlib介绍\n\n**Matplot...,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,FIGSIZE_TOO_SMALL;INSUFFICIENT_COLOR_CONTRAST;...,NaN
2,gulanaanwar/randomwalkgraph,rw_visual.py,2022-01-01,import matplotlib.pyplot as plt \n\nfrom rando...,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,MISSING_LEGEND;MISSING_TITLE;MISSING_XLABEL;MI...,NaN
3,harishcpu/CPU-Spy,graph.py,2022-01-01,import matplotlib.pyplot as pyp\r\nimport matp...,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,ANIMATIONS;MISSING_LEGEND;MISSING_TITLE;MISSIN...,NaN
4,wilsonleong/Gantt,chart.py,2022-01-02,"# -*- coding: utf-8 -*-\r\n""""""\r\nCreated on M...",1,1,1,0,1,0,...,0,0,0,0,0,0,0,0,FONTSIZE_TOO_SMALL;INSUFFICIENT_COLOR_CONTRAST...,NaN
